In [ ]:
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 77.0 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.16.1
    Uninstalling transformers-5.16.1:
      Successfully uninstalled transformers-5.16.1


In [ ]:
pip install -U "huggingface_hub"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 828.5/828.5 kB 16.1 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.29.0
    Uninstalling huggingface_hub-1.29.0:
      Successfully uninstalled huggingface_hub-1.29.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 5.17.0 requires huggingface-hub<2.0,>=1.5.0, but you have huggingface-hub 2.0.0 which is incompatible.
gradio-client 2.6.1 requires huggingface-hub<2.0,>=0.19.3, but you have huggingface-hub 2.0.0 which is incompatible.
tokenizers 0.23.1 requires huggingface-hub<2.0,>=0.16.4, but you have huggingface-hub 2.0.0 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 2.0.0 which is incompatible.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 2.0.0 whi

In [ ]:
import pandas as pd
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
manifesto_ids = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Thesis/manifesto_ids.csv")

In [ ]:
# Creating the manifesto id
manifesto_ids["manifesto_id"] = (
    manifesto_ids["party"].astype(str) + "_" + manifesto_ids["date"].astype(str)
)



In [ ]:
manifesto_ids.head(15)

,countryname,date,party,manifesto_id
0,Sweden,200209,11110,11110_200209
1,Sweden,200209,11220,11220_200209
2,Sweden,200209,11320,11320_200209
3,Sweden,200209,11420,11420_200209
4,Sweden,200209,11520,11520_200209
5,Sweden,200209,11620,11620_200209
6,Sweden,200209,11810,11810_200209


In [ ]:
len(manifesto_ids)

13

In [ ]:
from huggingface_hub import InferenceClient

from google.colab import userdata

hf_token = userdata.get("HF_token")

import time

import numpy as np
from sklearn.metrics import f1_score

from pathlib import Path

import random

import re

In [ ]:
client = InferenceClient(api_key=hf_token)

In [ ]:
# Defining the function for sending a request to the LLM


def classify_text(text):
    messages = [
        {
            "role": "system",
            "content": "You are a political manifesto classifier classifying Finnish text. Your task is to rate quasi-sentences as 1 if they are about mental health policy or psychiatric care, and 0 otherwise. You must ALWAYS first give your reasoning using a <think> tag and your final result as a binary number in a <result> tag.",
        },
        {
            "role": "user",
            "content": "<text>\nThere are many cases of homelessness in our country.\n</text>",
        },
        {
            "role": "assistant",
            "content": "<think>\nAlthough can be mental health related reasons behind homelessness, this statement doesn't mention them explicitly.\n</think>\n<result>\n0\n</result>",
        },
        {
            "role": "user",
            "content": "<text>\nSuicide is a leading cause of death amongst people aged between 25 and 50.\n</text>",
        },
        {
            "role": "assistant",
            "content": "<think>\nSuicides are closely associated with severe mental health issues.\n</think>\n<result>\n1\n</result>",
        },
        {
            "role": "user",
            "content": "<text>\nWe want to provide more support for families.\n</text>",
        },
        {
            "role": "assistant",
            "content": "<think>\nText does not explicitly mention mental health support.\n</think>\n<result>\n0\n</result>",
        },
        {
            "role": "user",
            "content": "<text>\nWe will increase the funding for children and adolescent psychiatric treatment.\n</text>",
        },
        {
            "role": "assistant",
            "content": "<think>\nText is about psychiatric care.\n</think>\n<result>\n1\n</result>",
        },
        {
            "role": "user",
            "content": "<text>\nOur party wants to increase the availability of counseling for asylum seekers.\n</text>",
        },
        {
            "role": "assistant",
            "content": "<think>\nText does not spesify that it means psychological counseling - could also be e.g. career counseling.\n</think>\n<result>\n0\n</result>",
        },
        {"role": "user", "content": f"<text>\n{text}\n</text>"},
        {
            "role": "assistant",
            "content": "<think>",
        },  # Prime the model to start thinking
    ]

    try:
        # Using a fast, instruction-tuned model
        response = client.chat_completion(
            model="meta-llama/Llama-4-Scout-17B-16E-Instruct",
            messages=messages,
            max_tokens=200,  # To allow thinking mode.
            temperature=0,  # Keep it consistent
        )

        content = response.choices[0].message.content
        match = re.search(r"<result>\n(\d+)\n</result>", content)

        # breakpoint()
        if match:
            # Convert the extracted text directly to an integer
            result = int(match.group(1))
        else:
            result = -1
        # To see if something was not classified at all
        return result
    except Exception as e:
        print(f"Error classifying: {e}")
        return None

    finally:
        time.sleep(random.uniform(0.05, 0.3))

In [ ]:
# Creating a function to open a file we need

def open_manifesto(manifesto_id):
    file_path = f"/content/drive/MyDrive/Colab Notebooks/Thesis/new_files/{manifesto_id}.csv"
    df = pd.read_csv(file_path)
    return df

In [ ]:
# a function to parse the coding into a summary table
def parse_summary(df):
    summary_row = pd.DataFrame(
        {
            "manifesto_id": df["manifesto_id"][2],
            "orig_language": df["language"][2],
            "total_rows": [len(df)],
            "rows_classified_1": [(df["predicted_score"] == 1).sum()],
        }
    )
    return summary_row

In [ ]:
# Creating the summary df
summary_df = pd.DataFrame(
    columns=["manifesto_id", "orig_language", "total_rows", "rows_classified_1"]
)

In [ ]:
# A test on a random manifesto

manifesto_id = manifesto_ids["manifesto_id"][16]
df = open_manifesto(manifesto_id)
df["predicted_score"] = df["text"].apply(classify_text)
df.to_csv(f"/content/drive/MyDrive/Colab Notebooks/Thesis/coded/{manifesto_id}_coded.csv",
        index=False,
    )

In [ ]:
num_manifestos = len(manifesto_ids["manifesto_id"])
print(num_manifestos)

13


In [ ]:
# Looping over the manifesto ids to encode with LLM

for i in range(0, num_manifestos):
    manifesto_id = manifesto_ids["manifesto_id"][i]
    print(f"Processing manifesto {i + 1} of {num_manifestos}")
    df = open_manifesto(manifesto_id)
    df["predicted_score"] = df["text"].apply(classify_text)
    df.to_csv(
        f"/content/drive/MyDrive/Colab Notebooks/Thesis/coded/{manifesto_id}_coded.csv",
        index=False,
    )
    summary_df = pd.concat([summary_df, parse_summary(df)], ignore_index=True)

Processing manifesto 1 of 13
Processing manifesto 2 of 13
Processing manifesto 3 of 13
Processing manifesto 4 of 13
Processing manifesto 5 of 13
Processing manifesto 6 of 13
Processing manifesto 7 of 13
Processing manifesto 8 of 13
Processing manifesto 9 of 13
Processing manifesto 10 of 13
Processing manifesto 11 of 13
Processing manifesto 12 of 13
Processing manifesto 13 of 13


In [ ]:
# Saving the summary
summary_df.to_csv(
    f"/content/drive/MyDrive/Colab Notebooks/Thesis/coded/summary.csv", index=False
)

In [ ]:
summary_df.head(10)

,manifesto_id,orig_language,total_rows,rows_classified_1
0,11110_200609,swedish,198,4
1,11110_200609,swedish,198,4
2,11220_200609,swedish,389,1
3,11320_200609,swedish,393,7
4,11420_200609,swedish,605,5
5,11520_200609,swedish,124,1
6,11620_200609,swedish,859,21
7,11810_200609,swedish,314,2
8,11110_201009,swedish,262,4
